## NASTRAN SOL 145 FLUTTER ANALYSIS AUTOMATION EXAMPLE

This notebook demonstrates the automation of NASTRAN for SOL 145 flutter analysis at Streamline Aerospace.\
<u>Prerequisites</u>
- Siemens FEMAP: v11.0.1
- NASTRAN Version: V8.5
- Python Version: 3.12.X

In [ ]:
#Import Statements
from flutter_nastran import (
    split_nastran_deck,
    flutter_card,
    generate_sol145_master_bdf,
    run_nastran,
    extract_vf_vg_data,
    repair_vf_vg_curves,
    make_vf_vg_plot,)
import numpy as np 
from pathlib import Path
import subprocess
import os

ModuleNotFoundError: No module named 'flutter_nastran.runner'

### Step 1 - Generating Bulk Data Files

Before running this notebook, ensure you have created an adequate structural and aerodynamic analysis model in FEMAP. Once completed, export this analysis model to a .bdf file using File>Export>Analysis Model. Once you've done this, update the filepath field in the code cell below to point to the .bdf file. Of note: Make sure the correct constraint set is referenced in the output analysis set.

In [2]:
#FEMAP export bdf model (Must contain aero and structural model)
working_dir = "C:/NASTRAN_FLUTTER" #name of directory 
os.chdir(working_dir)

analysis_model_bdfs = ["20261106_RV10Wing_00GAL_AILFIX.bdf",
                       "20261106_RV10Wing_00GAL_AILFREE.bdf",
                       "20261106_RV10Wing_30GAL_AILFIX.bdf",
                       "20261106_RV10Wing_30GAL_AILFREE.bdf",
                       "20261106_RV10Wing_41_5GAL_STIFFENED_AILFIX.bdf",
                       "20261106_RV10Wing_41_5GAL_STIFFENED_AILFREE.bdf"] #list of bdf files inside above working directory

#flutter sweep parameters 
altitude_ft = 15000 #altitude to survey flutter over 
Mach_min = 0.025
Mach_max = 0.8
Mach_num_pts = 32
spc_id = 3 #! make sure this is set (can actually be read off of input .bdf file)
ref_chord_in = 55
flutter_freq_interval = [0.002, 50] #interval to calculate reduced frequencies from
flutter_freq_max = 40 #maximum frequency for NASTRAN to survey up to 

In [ ]:
folder_paths = [] #store a list of paths to each sub-folder

for model_bdf in analysis_model_bdfs:

    #get analysis model bdf base name
    base_name = model_bdf.split("/")[-1] #remove any path /
    base_name = model_bdf.split(".")[0] #remove file extension

    #create a folder inside the working directory
    folder_path = working_dir + "/" + base_name
    os.makedirs(folder_path, exist_ok=True)
    folder_paths.append(folder_path)

    struc_path, aero_path = split_nastran_deck(input_bdf_path=model_bdf, output_dir=folder_path)

    flutter_path = flutter_card(altitude_ft=altitude_ft,
                    Mach_sweep=np.linspace(Mach_min, Mach_max, Mach_num_pts), 
                    freq_interval_Hz=flutter_freq_interval,
                    reference_chord_in=ref_chord_in,
                    output_dir=folder_path)

    run_file = generate_sol145_master_bdf(flutter_card_path=flutter_path,
                        struct_card_path=struc_path,
                        aero_card_path=aero_path,
                        base_title="RV10_WING",
                        output_dir=folder_path,
                        spc_id=spc_id,
                        max_freq_Hz=flutter_freq_max)

Separation complete:
  - Structural: C:/NASTRAN_FLUTTER/20261106_RV10Wing_00GAL_AILFIX/20261106_RV10Wing_00GAL_AILFIX_STRUCT.bdf
  - Aero: C:/NASTRAN_FLUTTER/20261106_RV10Wing_00GAL_AILFIX/20261106_RV10Wing_00GAL_AILFIX_AERO.bdf


Deck successfully generated: C:/NASTRAN_FLUTTER/20261106_RV10Wing_00GAL_AILFIX/flutter_15kft.bdf


Master SOL 145 deck generated: C:/NASTRAN_FLUTTER/20261106_RV10Wing_00GAL_AILFIX/xRUN_RV10_WING_flutter_15kft.bdf
Separation complete:
  - Structural: C:/NASTRAN_FLUTTER/20261106_RV10Wing_00GAL_AILFREE/20261106_RV10Wing_00GAL_AILFREE_STRUCT.bdf
  - Aero: C:/NASTRAN_FLUTTER/20261106_RV10Wing_00GAL_AILFREE/20261106_RV10Wing_00GAL_AILFREE_AERO.bdf


Deck successfully generated: C:/NASTRAN_FLUTTER/20261106_RV10Wing_00GAL_AILFREE/flutter_15kft.bdf


Master SOL 145 deck generated: C:/NASTRAN_FLUTTER/20261106_RV10Wing_00GAL_AILFREE/xRUN_RV10_WING_flutter_15kft.bdf
Separation complete:
  - Structural: C:/NASTRAN_FLUTTER/20261106_RV10Wing_30GAL_AILFIX/20261106_RV10Wing_30GAL_AILFIX_STRUCT.bdf
  - Aero: C:/NASTRAN_FLUTTER/20261106_RV10Wing_30GAL_AILFIX/20261106_RV10Wing_30GAL_AILFIX_AERO.bdf


Deck successfully generated: C:/NASTRAN_FLUTTER/20261106_RV10Wing_30GAL_AILFIX/flutter_15kft.bdf


... (3 outputs left out)

Deck successfully generated: C:/NASTRAN_FLUTTER/20261106_RV10Wing_41_5GAL_STIFFENED_AILFIX/flutter_15kft.bdf


Master SOL 145 deck generated: C:/NASTRAN_FLUTTER/20261106_RV10Wing_41_5GAL_STIFFENED_AILFIX/xRUN_RV10_WING_flutter_15kft.bdf
Separation complete:
  - Structural: C:/NASTRAN_FLUTTER/20261106_RV10Wing_41_5GAL_STIFFENED_AILFREE/20261106_RV10Wing_41_5GAL_STIFFENED_AILFREE_STRUCT.bdf
  - Aero: C:/NASTRAN_FLUTTER/20261106_RV10Wing_41_5GAL_STIFFENED_AILFREE/20261106_RV10Wing_41_5GAL_STIFFENED_AILFREE_AERO.bdf


Deck successfully generated: C:/NASTRAN_FLUTTER/20261106_RV10Wing_41_5GAL_STIFFENED_AILFREE/flutter_15kft.bdf


Master SOL 145 deck generated: C:/NASTRAN_FLUTTER/20261106_RV10Wing_41_5GAL_STIFFENED_AILFREE/xRUN_RV10_WING_flutter_15kft.bdf


### Step 2 - Running NASTRAN

Now that the suitable .bdf files have been generated. The NASTRAN executable can be called.

In [ ]:
nastran_path = Path(r"C:\FEMAPv1101\nastran\bin\nastran.exe")

for path in folder_paths: 

    #SEARCH FOR A FILE BEGINNING WITH xRUN
    search_path = Path(path)
    matching_files = list(search_path.glob("xRUN*.bdf"))
    try: 
        assert len(matching_files) == 1 
    except: 
        print("Unable to locate xRUN*.bdf File. Continuing")
        continue 

    input_bdf = matching_files[0]._str

    try:
        res = run.run_nastran(bdf_path=input_bdf,
                            nastran_path=nastran_path)
        print("NASTRAN execution completed successfully.")
        print("STDOUT:\n", res.stdout)
    except subprocess.CalledProcessError as err:
        print(f"NASTRAN run failed with return code {err.returncode}")
        print("STDERR:\n", err.stderr)
        print("STDOUT:\n", err.stdout)

### Step 3 - Postprocessing
Vf-Vg diagrams can be plotted using results found the in the generated .f06 files. The following cell creates matplotlib Vf-Vg plots for all .f06 files where results exist.

In [ ]:
for path in folder_paths: 

    #search for .f06 file
    search_path = Path(path)
    matching_files = list(search_path.glob("*.f06"))
    try: 
        assert len(matching_files) == 1 
    except: 
        print("Unable to locate .f06 File. Continuing")
        continue 

    f06 = matching_files[0]._str

    raw_data = post.extract_vf_vg_data(f06)
    clean_data = post.repair_vf_vg_curves(raw_data)
    
    # Plots structural modes (filtering rigid body modes < 1.0 Hz)
    # and annotates the lowest flutter crossing at g = 0.03
    post.make_vf_vg_plot(
        clean_data,
        suptitle=f06,
        mode_floor_Hz=0.001,
        damping_threshold=0.03,
        save_plot=True)

## Step 4 - Complex Mode Shape Output

Once you have determined if flutter is present within the Mach envelopes surveyed above, NASTRAN can be used to generate a complex mode shape output. In the Mach_numbers list, enter the mach numbers where the onset of unstable flutter it first observed. This should be at a damping threshold of +0.01 to +0.03 for most aircraft structures.

In [4]:
#declare mach numbers to check 
Mach_numbers = [0.56, 0.23, 0.39, 0.23, 0.28, 0.22] #mach numbers where onset of flutter is expected. Obtained from Vf-Vg diagrams

In [ ]:
#make sure Mach_numbers list is the correct length
assert len(Mach_numbers) == len(analysis_model_bdfs)
complex_folder_paths = []

for i,model_bdf in enumerate(analysis_model_bdfs):

    #get analysis model bdf base name
    base_name = model_bdf.split("/")[-1] #remove any path /
    base_name = model_bdf.split(".")[0] #remove file extension

    #create a folder inside the working directory
    folder_path = working_dir + "/" + base_name + "_COMPLEX"
    os.makedirs(folder_path, exist_ok=True)
    complex_folder_paths.append(folder_path)

    struc_path, aero_path = bdf.split_nastran_deck(input_bdf_path=model_bdf, output_dir=folder_path)

    flutter_path = bdf.flutter_card(altitude_ft=altitude_ft,
                    Mach_sweep=np.array([Mach_numbers[i]]), 
                    freq_interval_Hz=flutter_freq_interval,
                    reference_chord_in=ref_chord_in,
                    output_dir=folder_path,
                    complex_mode_output=True)

    run_file = bdf.generate_sol145_master_bdf(flutter_card_path=flutter_path,
                        struct_card_path=struc_path,
                        aero_card_path=aero_path,
                        base_title="RV10_WING",
                        output_dir=folder_path,
                        spc_id=spc_id,
                        max_freq_Hz=flutter_freq_max,
                        complex_mode_output=True)

Separation complete:
  - Structural: C:/NASTRAN_FLUTTER/20261106_RV10Wing_00GAL_AILFIX_COMPLEX/20261106_RV10Wing_00GAL_AILFIX_STRUCT.bdf
  - Aero: C:/NASTRAN_FLUTTER/20261106_RV10Wing_00GAL_AILFIX_COMPLEX/20261106_RV10Wing_00GAL_AILFIX_AERO.bdf


Deck successfully generated: C:/NASTRAN_FLUTTER/20261106_RV10Wing_00GAL_AILFIX_COMPLEX/flutter_15kft_complex.bdf


Master SOL 145 deck generated: C:/NASTRAN_FLUTTER/20261106_RV10Wing_00GAL_AILFIX_COMPLEX/xRUN_RV10_WING_flutter_15kft_complex.bdf
Separation complete:
  - Structural: C:/NASTRAN_FLUTTER/20261106_RV10Wing_00GAL_AILFREE_COMPLEX/20261106_RV10Wing_00GAL_AILFREE_STRUCT.bdf
  - Aero: C:/NASTRAN_FLUTTER/20261106_RV10Wing_00GAL_AILFREE_COMPLEX/20261106_RV10Wing_00GAL_AILFREE_AERO.bdf


Deck successfully generated: C:/NASTRAN_FLUTTER/20261106_RV10Wing_00GAL_AILFREE_COMPLEX/flutter_15kft_complex.bdf


Master SOL 145 deck generated: C:/NASTRAN_FLUTTER/20261106_RV10Wing_00GAL_AILFREE_COMPLEX/xRUN_RV10_WING_flutter_15kft_complex.bdf
Separation complete:
  - Structural: C:/NASTRAN_FLUTTER/20261106_RV10Wing_30GAL_AILFIX_COMPLEX/20261106_RV10Wing_30GAL_AILFIX_STRUCT.bdf
  - Aero: C:/NASTRAN_FLUTTER/20261106_RV10Wing_30GAL_AILFIX_COMPLEX/20261106_RV10Wing_30GAL_AILFIX_AERO.bdf


Deck successfully generated: C:/NASTRAN_FLUTTER/20261106_RV10Wing_30GAL_AILFIX_COMPLEX/flutter_15kft_complex.bdf


... (3 outputs left out)

Deck successfully generated: C:/NASTRAN_FLUTTER/20261106_RV10Wing_41_5GAL_STIFFENED_AILFIX_COMPLEX/flutter_15kft_complex.bdf


Master SOL 145 deck generated: C:/NASTRAN_FLUTTER/20261106_RV10Wing_41_5GAL_STIFFENED_AILFIX_COMPLEX/xRUN_RV10_WING_flutter_15kft_complex.bdf
Separation complete:
  - Structural: C:/NASTRAN_FLUTTER/20261106_RV10Wing_41_5GAL_STIFFENED_AILFREE_COMPLEX/20261106_RV10Wing_41_5GAL_STIFFENED_AILFREE_STRUCT.bdf
  - Aero: C:/NASTRAN_FLUTTER/20261106_RV10Wing_41_5GAL_STIFFENED_AILFREE_COMPLEX/20261106_RV10Wing_41_5GAL_STIFFENED_AILFREE_AERO.bdf


Deck successfully generated: C:/NASTRAN_FLUTTER/20261106_RV10Wing_41_5GAL_STIFFENED_AILFREE_COMPLEX/flutter_15kft_complex.bdf


Master SOL 145 deck generated: C:/NASTRAN_FLUTTER/20261106_RV10Wing_41_5GAL_STIFFENED_AILFREE_COMPLEX/xRUN_RV10_WING_flutter_15kft_complex.bdf


In [ ]:
for path in complex_folder_paths: 

    #SEARCH FOR A FILE BEGINNING WITH xRUN
    search_path = Path(path)
    matching_files = list(search_path.glob("xRUN*.bdf"))
    try: 
        assert len(matching_files) == 1 
    except: 
        print("Unable to locate xRUN*.bdf File. Continuing")
        continue 

    input_bdf = matching_files[0]._str

    try:
        res = run.run_nastran(bdf_path=input_bdf,
                            nastran_path=nastran_path)
        print("NASTRAN execution completed successfully.")
        print("STDOUT:\n", res.stdout)
    except subprocess.CalledProcessError as err:
        print(f"NASTRAN run failed with return code {err.returncode}")
        print("STDERR:\n", err.stderr)
        print("STDOUT:\n", err.stdout)

Now that NASTRAN has run succesfully, reimport the solution .op2 file back into FEMAP using file>import>analysis results. This should import multiple mode results sets. Model>Output>Expand Complex on the offending mode to generate multiple phase angle output results sets. Highlight all, right-click, and click animate multiset. A .gif file can now be saved showing the behavior of the structure at the onset of unstable flutter. 